# Controle de dados e modelo: DVC e MLflow

**Pós Tech · Machine Learning Engineering · Live de 04/11/2026**

Neste notebook vamos versionar um dataset com **DVC** e registrar experimentos e modelos com **MLflow**, e no final ligar os dois.

**Antes de começar**
- Salve uma cópia: *Arquivo → Salvar uma cópia no Drive*.
- Rode as células **na ordem**, de cima para baixo.
- Linhas que começam com `!` rodam no **terminal** do Colab.
- A pasta `/content` **some quando o Colab desconecta**. Se isso acontecer, rode tudo de novo desde o início (leva uns 5 minutos).
- Quer **recomeçar**? Rode a **célula 2** de novo: ela apaga o projeto anterior e começa limpo. (Se o Colab pedir para reiniciar a sessão depois da célula 1, clique em reiniciar e rode **a partir da célula 2**.)

## Parte 0 — Preparação

In [ ]:
# Célula 1 — instalar as bibliotecas (leva cerca de 1 minuto)
!pip install -q dvc "mlflow>=3.1" scikit-learn pandas

In [ ]:
# Célula 2 — identidade do Git e pasta do projeto (rodar de novo recomeça o projeto do zero)
import os, shutil

!git config --global user.email "aluno@exemplo.com"
!git config --global user.name "Aluno"
!git config --global init.defaultBranch main

ROOT = "/content" if os.path.isdir("/content") else os.path.abspath("sim_colab")
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

# apaga o projeto e o remote de tentativas anteriores, para sempre começar limpo
for pasta in ("projeto-churn", "dvc-remote"):
    shutil.rmtree(f"{ROOT}/{pasta}", ignore_errors=True)

os.makedirs(f"{ROOT}/projeto-churn")
os.chdir(f"{ROOT}/projeto-churn")
print("Pasta de trabalho:", os.getcwd())

## Parte A — DVC

### A1. Iniciar Git e DVC
O `dvc init` cria a pasta `.dvc/` (configuração) e o `.dvcignore`.

In [ ]:
!git init

# só inicia o DVC se ainda não foi iniciado (assim a célula pode ser rodada de novo)
if not os.path.isdir(".dvc"):
    !dvc init
else:
    print("DVC já iniciado nesta pasta. Para recomeçar do zero, rode a célula 2.")

!git commit -m "inicia projeto com dvc"

In [ ]:
!ls -a

### A2. Dados v1 (5.000 linhas)
Baixamos o dataset *Telco Customer Churn* e salvamos só as 5.000 primeiras linhas. Essa é a versão 1.

In [ ]:
import pandas as pd

os.makedirs("data/raw", exist_ok=True)

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
df.iloc[:5000].to_csv("data/raw/churn.csv", index=False)   # v1: só as 5.000 primeiras
print(df.shape)

In [ ]:
!dvc add data/raw/churn.csv

O CSV foi para o **cache** do DVC. O Git passa a **ignorar** o CSV e guarda só o **ponteiro** `churn.csv.dvc`, com o hash MD5.

In [ ]:
!cat data/raw/churn.csv.dvc
!cat data/raw/.gitignore

In [ ]:
!git add data/raw/churn.csv.dvc data/raw/.gitignore
!git commit -m "dados v1: 5000 linhas"
!git tag data-v1

### A3. Remote (pasta) e push
O remote é onde o dado de verdade fica guardado. Aqui usamos uma pasta; num projeto real seria um bucket (S3, GCS...).

In [ ]:
!dvc remote add -d storage {ROOT}/dvc-remote
!git add .dvc/config
!git commit -m "configura remote do dvc"
!dvc push

In [ ]:
!find {ROOT}/dvc-remote -type f

### A4. Chegaram dados novos (v2, 7.043 linhas)
Mesmo caminho de arquivo, conteúdo novo. O Git só vai ver o **hash** mudar.

In [ ]:
df.to_csv("data/raw/churn.csv", index=False)   # v2: base completa

In [ ]:
!dvc add data/raw/churn.csv
!git add data/raw/churn.csv.dvc
!git commit -m "dados v2: base completa"
!git tag data-v2
!dvc push

In [ ]:
!git diff data-v1 data-v2 -- data/raw/churn.csv.dvc

### A5. Viagem no tempo
Antes de rodar: quantas linhas você acha que o arquivo terá depois do `checkout`? (O `wc -l` conta uma linha a mais por causa do cabeçalho.)

In [ ]:
!wc -l data/raw/churn.csv          # v2: 7.044

In [ ]:
!git checkout -q data-v1
!dvc checkout
!wc -l data/raw/churn.csv          # voltou para v1: 5.001

In [ ]:
!git checkout -q main
!dvc checkout
!wc -l data/raw/churn.csv          # v2 de novo: 7.044

### A6. Desastre e recuperação
Apagamos o dado **e** o cache local. O `dvc pull` busca tudo de volta do remote.

In [ ]:
!rm -rf data/raw/churn.csv .dvc/cache
!ls data/raw

In [ ]:
!dvc pull
!wc -l data/raw/churn.csv

## Parte B — MLflow

### B1. Preparar os dados (já versionados pelo DVC)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, yaml
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn", "customerID"])
cat_cols = X.select_dtypes("object").columns.tolist()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# A "impressão digital" da versão de dados que estamos usando (vem do DVC)
data_md5 = yaml.safe_load(open("data/raw/churn.csv.dvc"))["outs"][0]["md5"]
print(data_md5)

### B2. Configurar o MLflow
Usamos um banco SQLite local, necessário para o Model Registry.

In [ ]:
import mlflow, mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn")

# silencia um aviso informativo do MLflow 3 ao registrar o modelo (não afeta o resultado)
import logging
logging.getLogger("mlflow.tracking._model_registry.fluent").setLevel(logging.ERROR)

### B3. Função de treino com tracking
- `log_params`: o que eu escolhi
- `log_metrics`: o que eu obtive
- `set_tag`: contexto (aqui, a versão dos dados)
- `log_model`: o artefato, com assinatura de entrada e saída

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def treinar(nome, modelo, params):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
                            remainder="passthrough")
    pipe = Pipeline([("pre", pre), ("clf", modelo)])

    with mlflow.start_run(run_name=nome) as run:
        mlflow.log_params(params)
        mlflow.set_tag("data_md5", data_md5)        # <- ponte com o DVC

        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        proba = pipe.predict_proba(Xte)[:, 1]

        mlflow.log_metrics({
            "accuracy": accuracy_score(yte, pred),
            "f1": f1_score(yte, pred),
            "roc_auc": roc_auc_score(yte, proba),
        })
        mlflow.sklearn.log_model(pipe, name="model",
                                 signature=infer_signature(Xte, pred),
                                 input_example=Xte.head(3),
                                 serialization_format="cloudpickle")   # evita o erro "untrusted types" do skops
        return run.info.run_id

### B4. Três experimentos

In [ ]:
treinar("logreg",  LogisticRegression(max_iter=1000),
        {"modelo": "logreg", "max_iter": 1000})
treinar("rf-prof5", RandomForestClassifier(max_depth=5, random_state=42),
        {"modelo": "rf", "max_depth": 5})
treinar("rf-prof15", RandomForestClassifier(max_depth=15, random_state=42),
        {"modelo": "rf", "max_depth": 15})

### B5. Comparar as runs
Ordenamos pela métrica `roc_auc` (AUC: *Area Under the Curve*).

In [ ]:
runs = mlflow.search_runs(order_by=["metrics.roc_auc DESC"])
runs[["tags.mlflow.runName", "params.modelo", "params.max_depth",
      "metrics.accuracy", "metrics.f1", "metrics.roc_auc", "tags.data_md5"]]

#### (Opcional) Abrir a interface do MLflow no Colab
> **Demonstração da professora.** As duas opções abaixo ainda não foram testadas no Colab; se uma não abrir, a tabela acima já mostra o essencial.

**Opção 1 — iframe do próprio Colab** (sem conta e sem download):

In [ ]:
# Sobe a interface do MLflow em segundo plano e exibe dentro do notebook
import subprocess, sys, time
ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db", "--port", "5000"],
                      stdout=open("mlflow_ui.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=700)
except ImportError:
    print("Fora do Colab: abra http://localhost:5000 no navegador.")

**Opção 2 — túnel com `cloudflared`** (baixa um binário oficial da Cloudflare; só use se a opção 1 falhar):

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import subprocess, time
tunel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
                         stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' cloudflared.log | head -1

### B6. Registrar o melhor modelo e dar um alias
O alias `champion` aponta para a versão que está "em produção". Para trocar o modelo, basta **mover o alias**.

In [ ]:
melhor = runs.iloc[0].run_id
mv = mlflow.register_model(f"runs:/{melhor}/model", "churn-model")
MlflowClient().set_registered_model_alias("churn-model", "champion", mv.version)
print("versão registrada:", mv.version)

In [ ]:
modelo = mlflow.pyfunc.load_model("models:/churn-model@champion")
modelo.predict(Xte.head(5))

### B7. Explorando o Registry: versões, tags e descrição
Cada versão registrada guarda de onde veio (a run), **aliases**, **tags** e uma **descrição**. Vamos documentar a versão 1 e criar uma função para listar tudo.

In [ ]:
client = MlflowClient()
client.set_model_version_tag("churn-model", mv.version, "data_md5", data_md5)
client.set_model_version_tag("churn-model", mv.version, "validado", "sim")
client.update_model_version("churn-model", mv.version,
                            description=f"Melhor run por roc_auc: {runs.iloc[0]['tags.mlflow.runName']}")

def mostrar_versoes():
    aliases = client.get_registered_model("churn-model").aliases     # {alias: versão}
    linhas = []
    for v in client.search_model_versions("name='churn-model'"):
        linhas.append({"versão": int(v.version),
                       "aliases": [a for a, ver in aliases.items() if ver == v.version],
                       "run": v.run_id[:8], "validado": v.tags.get("validado", "-")})
    return pd.DataFrame(linhas).sort_values("versão").reset_index(drop=True)

mostrar_versoes()

> Na interface do MLflow, o menu **Models** mostra as mesmas informações: versões, aliases e tags.

### B8. Um challenger entra em cena
Treinamos um novo modelo, registramos como **versão 2** e damos o alias `challenger`. O `champion` continua na versão 1: nada mudou em produção.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

rid = treinar("gb-challenger",
              GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
              {"modelo": "gb", "n_estimators": 200, "learning_rate": 0.05, "max_depth": 3})

v2 = mlflow.register_model(f"runs:/{rid}/model", "churn-model")
client.set_model_version_tag("churn-model", v2.version, "data_md5", data_md5)
client.set_registered_model_alias("churn-model", "challenger", v2.version)
mostrar_versoes()

### B9. Champion x challenger
A decisão de promover vem de uma **comparação**, não de um palpite.

In [ ]:
def auc_do_alias(alias):
    v = client.get_model_version_by_alias("churn-model", alias)
    return v.version, mlflow.get_run(v.run_id).data.metrics["roc_auc"]

vc, auc_c = auc_do_alias("champion")
vh, auc_h = auc_do_alias("challenger")
print(f"champion   (v{vc}): roc_auc = {auc_c:.4f}")
print(f"challenger (v{vh}): roc_auc = {auc_h:.4f}")
print("diferença:", round(auc_h - auc_c, 4))

### B10. Promover: mover o alias
Na vida real, só promovemos se a comparação aprovar (por exemplo, com uma margem mínima de ganho). Aqui promovemos para ver o **mecanismo**: o `champion` passa a apontar para a versão 2, e guardamos a versão anterior para um eventual rollback.

In [ ]:
anterior = vc
client.set_registered_model_alias("churn-model", "champion", vh)
print("champion agora é a versão", client.get_model_version_by_alias("churn-model", "champion").version)
mostrar_versoes()

In [ ]:
# quem consome o modelo não mudou uma linha: o mesmo endereço, outro modelo
modelo = mlflow.pyfunc.load_model("models:/churn-model@champion")
modelo.predict(Xte.head(5))

### B11. Rollback: mover o alias de volta
Algo deu errado em produção? Voltar é mover o alias para a versão anterior. Nenhum arquivo é copiado.

In [ ]:
client.set_registered_model_alias("churn-model", "champion", anterior)
print("champion voltou para a versão", client.get_model_version_by_alias("churn-model", "champion").version)
mostrar_versoes()

## Parte C — Juntando tudo
Do modelo `champion` chegamos à run, e da run chegamos à versão dos dados.

In [ ]:
champ = MlflowClient().get_model_version_by_alias("churn-model", "champion")
run = mlflow.get_run(champ.run_id)
print("Modelo champion -> run:", champ.run_id)
print("Treinado com dados (md5):", run.data.tags["data_md5"])

In [ ]:
# E esse hash é o mesmo que o DVC tem para a versão atual dos dados
!grep md5 data/raw/churn.csv.dvc
# Para recuperar exatamente esses dados em qualquer máquina:
#   git checkout data-v2 && dvc checkout   (ou dvc pull)

**Reprodutibilidade fechada:** modelo → run → hash dos dados → versão no Git → dado no remote.

---
### Problemas comuns
| Sintoma | Solução |
|---|---|
| `dvc: command not found` | rodar a célula 1 de novo |
| `git commit` pede identidade | rodar a célula 2 |
| `ModuleNotFoundError` depois de reconectar | o Colab reiniciou: rodar as células 1 e 2 e refazer o projeto |
| Registry reclama do store | usar `sqlite:///mlflow.db` (B2) |
| Resultados diferentes entre alunos | manter `random_state=42` |